In [0]:
%sql DESCRIBE mf_orders;

creates a persistent Delta table containing completed orders only.

In [0]:
%sql

CREATE OR REPLACE TABLE mf_successful AS

SELECT
    third_party_id,
    groww_order_id,
    created_at,
    order_completed_at,
    order_type,
    order_kind,
    order_subkind,
    buy_sell,
    buy_sell_type,
    sip_flag,
    is_liquid,
    is_nfo,
    is_first_order,
    order_val,
    placed_amount,
    qty,
    scheme_name,
    scheme_code,
    isin,
    payment_source,
    payment_status,
    total_payment_attempts,
    brokerage_inr

FROM mf_orders

WHERE UPPER(TRIM(order_status)) = 'COMPLETED';

aggregate the orders by user.

In [0]:
%sql

CREATE OR REPLACE TABLE mf_user_metrics AS

SELECT

    third_party_id,

    -- Revenue
    COUNT(DISTINCT groww_order_id) * 2
        AS total_brokerage,

    -- Investment activity
    COUNT(DISTINCT groww_order_id)
        AS total_completed_orders,

    SUM(COALESCE(order_val, 0))
        AS total_order_value,

    AVG(order_val)
        AS avg_order_value,

    COUNT(DISTINCT TO_DATE(created_at))
        AS active_investment_days,

    -- Product diversification
    COUNT(DISTINCT scheme_code)
        AS distinct_schemes,

    COUNT(DISTINCT isin)
        AS distinct_isins,

    -- Investment behaviour
    SUM(CASE WHEN sip_flag = 1
        THEN 1 ELSE 0 END) AS sip_orders,

    SUM(CASE WHEN is_liquid = 1
        THEN 1 ELSE 0 END) AS liquid_fund_orders,

    SUM(CASE WHEN is_nfo = 1
        THEN 1 ELSE 0 END) AS nfo_orders,

    SUM(CASE WHEN is_first_order = TRUE
        THEN 1 ELSE 0 END) AS first_order_count,

    SUM(CASE WHEN buy_sell = 'BUY'
        THEN 1 ELSE 0 END) AS buy_orders,

    SUM(CASE WHEN buy_sell = 'SELL'
        THEN 1 ELSE 0 END) AS sell_orders,

    MAX(order_completed_at)
        AS last_order_date

FROM mf_successful

GROUP BY third_party_id;

Identify high-value MF users
Because the revenue is fixed at ₹2 per completed order, users generating more completed orders also generate more MF revenue.

In [0]:
%sql

CREATE OR REPLACE TABLE mf_user_segments AS

SELECT
    *,
    NTILE(5) OVER (
        ORDER BY total_brokerage DESC
    ) AS value_segment

FROM mf_user_metrics;

In [0]:
%sql

SELECT

    value_segment,

    COUNT(*) AS total_users,

    ROUND(AVG(total_brokerage), 2) AS avg_revenue,

    SUM(total_brokerage) AS total_revenue,

    ROUND(
        100.0 * SUM(total_brokerage) /
        NULLIF(SUM(SUM(total_brokerage)) OVER (), 0),
        2
    ) AS revenue_contribution_pct

FROM mf_user_segments

GROUP BY value_segment

ORDER BY value_segment;

Additional analysis: SIP adoption
This tells us what percentage of high-value and other MF users have placed at least one SIP order.

In [0]:
%sql

SELECT

    CASE
        WHEN value_segment = 1 THEN 'High Value'
        ELSE 'Other Users'
    END AS user_category,

    COUNT(*) AS total_users,

    COUNT_IF(sip_orders > 0) AS sip_users,

    ROUND(
        100.0 * COUNT_IF(sip_orders > 0) / COUNT(*),
        2
    ) AS sip_adoption_pct

FROM mf_user_segments

GROUP BY 1;

In [0]:
%sql

SELECT
    buy_sell,
    COUNT(*) AS total_orders
FROM mf_successful
GROUP BY buy_sell;